# 7. Zone-entry outcomes at pace


### Purpose

Assess whether faster puck movement before a zone entry produces a more successful attack. Compare carried, played, and dumped entries separately: shots after carried/played entries and puck recovery after dumps.

### Expected outcome

An entry-level research table, pooled preceding-pace tertiles, and five-second outcome comparisons with coverage and team context at 5v5. Identify entry methods and sequences worth coaching review without assuming higher pace causes better outcomes.


## 7.1. Setup and load processed data

Load the augmented events, transition pace, and reference tables used in notebook 05. Recorded clocks and estimated modeled timestamps retain their original meaning. Only 5v5 entries are included. Possession sequences already split at manpower changes, so preceding movement belongs to the same 5v5 sequence.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


## 7.2. Define entries and five-second outcomes

Count only 5v5 `Zone Entry` rows, using `Carried`, `Played`, and `Dumped` subtypes. Do not count `Dump In/Out` rows again as entries.

For carried/played entries, success is a subsequent `Shot` or `Goal` by the entering team within five seconds, before opponent control, a faceoff restart, a goal, or another entry. A shot attempt includes blocked and missed shots. For dumped entries, inspect the first subsequent recorded `Puck Recovery` within five seconds, before a restart, goal, or another entry. Success means that recovery belongs to the entering team. Recovery can occur in a different inferred possession sequence; opponent control alone does not end the dump-recovery search. This measures first recorded recovery, not continuous retention or eventual recovery.

Use event order and modeled timestamps; subsequent same-time events can qualify. The focal entry cannot be an outcome. Require five seconds of observed period coverage for primary comparisons, retaining incomplete follow-up in the coverage table. Stop outcome attribution when play leaves 5v5. No qualifying outcome before that boundary in a complete observed window is a failure. Penalty-shot goals are not shot attempts, and penalty records alone do not establish a stoppage.


In [ ]:
OUTCOME_SECONDS = 5
assert OUTCOME_SECONDS > 0
entries = events.loc[
    events["event"].eq("Zone Entry") & events["is_5v5"]
].copy()
entries["entry_type"] = entries["detail_1"]
assert entries["is_5v5"].all()
assert entries["entry_type"].isin(["Carried", "Played", "Dumped"]).all()
entries["team_name"] = entries["team_id"].map(teams.set_index("team_id")["team_name"])
entries["situation"] = entries["team_skaters"].astype(str) + "v" + entries["opponent_skaters"].astype(str)
entries["game_date"] = entries["game_id"].map(games.set_index("game_id")["game_date"])
period_last_clock = events.groupby(["game_id", "period"])["modeled_clock_seconds"].min()
period_events = {key: group for key, group in events.groupby(["game_id", "period"], sort=False)}
control_events = {"Puck Recovery", "Takeaway", "Faceoff Win", "Pass", "Pass Reception",
                  "Incomplete Pass", "Shot", "Goal", "Dump In/Out", "Zone Entry"}
outcome_rows = []
for entry in entries.itertuples(index=False):
    following = period_events[(entry.game_id, entry.period)]
    following = following.loc[
        following["event_id"].gt(entry.event_id)
        & following["modeled_clock_seconds"].le(entry.modeled_clock_seconds)
        & following["modeled_clock_seconds"].ge(entry.modeled_clock_seconds - OUTCOME_SECONDS)
    ]
    complete_followup = (
        entry.modeled_clock_seconds - period_last_clock.loc[(entry.game_id, entry.period)] >= OUTCOME_SECONDS
    )
    outcome_event_id = None
    recovered_by_team_id = None
    successful = False
    for event in following.itertuples(index=False):
        if not event.is_5v5:
            break
        if event.event in {"Faceoff Win", "Penalty Shot Goal", "Zone Entry"}:
            break
        if entry.entry_type == "Dumped":
            if event.event == "Puck Recovery":
                outcome_event_id = event.event_id
                recovered_by_team_id = event.team_id
                successful = event.team_id == entry.team_id
                break
            if event.event == "Goal":
                break
        else:
            if event.event in control_events and event.team_id != entry.team_id:
                break
            if event.event in {"Shot", "Goal"} and event.team_id == entry.team_id:
                successful = True
                outcome_event_id = event.event_id
                break
    outcome_rows.append({
        "event_id": entry.event_id, "complete_followup": bool(complete_followup),
        "outcome_success": float(successful) if complete_followup else np.nan,
        "outcome_event_id": outcome_event_id, "recovered_by_team_id": recovered_by_team_id,
    })
entries = entries.merge(pd.DataFrame(outcome_rows), on="event_id", validate="one_to_one")
display(entries.groupby(["entry_type"]).agg(
    entries=("event_id", "size"), complete_followup=("complete_followup", "sum"),
    outcome_rate=("outcome_success", "mean"),
))


## 7.3. Calculate sequence pace preceding entry

Pool distance and modeled movement duration across the entry's inferred possession sequence, using only transitions whose end event ID precedes the entry. There is no fixed lookback. Event order includes earlier same-time events, while movements ending at the entry and all later movement are excluded.

Use the existing possession IDs and transition pace inputs. Do not use the full exported sequence pace directly: it can include movement after the entry and would mix the predictor with subsequent outcomes. Sequence boundaries still reflect possession, manpower, and restart rules from notebook 02.

Unknown or zero-duration preceding pace is excluded from tertiles and reported in coverage, rather than assigned zero. Retain movement duration, count, and available history span. Modeled timestamps are estimates, and source-specific entry locations can affect available approach movement. The outcome window remains five seconds.


In [ ]:
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()].copy()
# Reuse existing possession sequences; trim each one at the entry event.
movement_groups = {
    pid: group[["end_event_id", "distance_ft", "modeled_elapsed_seconds"]].to_numpy(dtype=float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
sequence_start = events.groupby("possession_id")["modeled_clock_seconds"].max()
pace_rows = []
for entry in entries.itertuples(index=False):
    movements = movement_groups.get(entry.possession_id)
    if movements is not None:
        movements = movements[movements[:, 0] < entry.event_id]
    count = 0 if movements is None else len(movements)
    distance = 0.0 if movements is None else float(movements[:, 1].sum())
    duration = 0.0 if movements is None else float(movements[:, 2].sum())
    pace_rows.append({
        "event_id": entry.event_id,
        "preceding_transitions": count,
        "preceding_distance_ft": distance,
        "preceding_movement_seconds": duration,
        "preceding_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "available_history_seconds": float(
            sequence_start.loc[entry.possession_id] - entry.modeled_clock_seconds
        ),
        "pace_status": "eligible" if duration > 0 else "no_history" if count == 0 else "zero_duration",
    })
entries = entries.merge(pd.DataFrame(pace_rows), on="event_id", validate="one_to_one")
coverage = entries.groupby("pace_status").agg(
    entries=("event_id", "size"), outcome_rate=("outcome_success", "mean")
).reset_index()
display(coverage)


## 7.4. Define pooled pace tertiles

Define three equal-frequency bands (slower, middle, faster) from all entries with measurable preceding pace across all sources combined, sharing boundaries across entry types and teams. Keep tied values together. These are descriptive boundaries, not thresholds for a coaching instruction. The outcome analysis additionally requires complete five-second follow-up.

Source provenance remains in the entry records, but is not used for grouping or adjustment. Pooled patterns may reflect differences in competition and recording practices.


In [ ]:
eligible_entries = entries.loc[entries["pace_status"].eq("eligible")].copy()
bins, edges = pd.qcut(eligible_entries["preceding_pace_ft_s"], q=3,
                      labels=False, retbins=True, duplicates="drop")
eligible_entries["pace_tertile"] = (bins + 1).astype("Int64")
tertile_boundaries = pd.DataFrame({
    "pace_tertile": np.arange(1, len(edges)),
    "lower_pace_ft_s": edges[:-1], "upper_pace_ft_s": edges[1:],
})
tertile_boundaries["pace_band"] = tertile_boundaries["pace_tertile"].map(
    {1: "Slower", 2: "Middle", 3: "Faster"}
)
eligible_entries["pace_band"] = eligible_entries["pace_tertile"].map(
    {1: "Slower", 2: "Middle", 3: "Faster"}
)
display(tertile_boundaries.round(2))
analysis_entries = eligible_entries.loc[eligible_entries["complete_followup"]].copy()
display(entries.groupby(["entry_type", "pace_status", "complete_followup"])
        .size().rename("entries").reset_index())


## 7.5. Compare outcomes across pace tertiles

Plot shot-attempt probability for carried/played entries and entering-team first-recovery probability for dumps. These outcomes have different meanings and should not be ranked against each other as one common success measure. Show how entry selection varies across tertiles alongside the conditional outcomes.

Show outcomes as bars without uncertainty intervals. Counts and contributing games remain available in the table and hover. Repeated teams, sparse played-entry samples, and only 34 games limit interpretation.


In [ ]:
entry_summary = analysis_entries.groupby(
    ["entry_type", "pace_tertile"], observed=True
).agg(entries=("event_id", "size"), successes=("outcome_success", "sum"),
      games=("game_id", "nunique"), success_rate=("outcome_success", "mean"),
      median_pace_ft_s=("preceding_pace_ft_s", "median")).reset_index()
entry_summary["success_pct"] = 100 * entry_summary["success_rate"]

entry_summary["pace_band"] = entry_summary["pace_tertile"].map(
    {1: "Slower", 2: "Middle", 3: "Faster"}
)
display(entry_summary.round(2))

entry_types = ["Carried", "Played", "Dumped"]
entry_fig = make_subplots(rows=1, cols=3, subplot_titles=entry_types)
for col, kind in enumerate(entry_types, start=1):
    summary = entry_summary.loc[entry_summary["entry_type"].eq(kind)]
    entry_fig.add_trace(go.Bar(
        x=summary["pace_band"], y=summary["success_pct"], showlegend=False,
        marker_color=["#93c5fd", "#3b82f6", "#1d4ed8"],
        customdata=summary[["entries", "games", "median_pace_ft_s"]].to_numpy(),
        hovertemplate=("%{x}<br>Outcome: %{y:.1f}%<br>Entries: %{customdata[0]}"
                       "<br>Games: %{customdata[1]}<br>Median pace: %{customdata[2]:.1f} ft/s<extra></extra>"),
    ), row=1, col=col)
    entry_fig.update_yaxes(range=[0,100], title_text="First recovery (%)" if kind=="Dumped" else "Shot attempt (%)", row=1,col=col)
entry_fig.update_layout(title=f"5v5 entry outcomes within {OUTCOME_SECONDS}s | preceding possession-sequence pace",
                        template="plotly_white",height=450)
entry_fig.update_xaxes(title_text="Preceding pace", categoryorder="array",
                       categoryarray=["Slower", "Middle", "Faster"])
entry_fig.show()

entry_selection = eligible_entries.groupby(["pace_tertile", "entry_type"], observed=True).size().rename("entries").reset_index()
entry_selection["pace_band"] = entry_selection["pace_tertile"].map({1: "Slower", 2: "Middle", 3: "Faster"})
entry_selection["share_pct"] = 100 * entry_selection["entries"] / entry_selection.groupby(
    ["pace_tertile"])["entries"].transform("sum")
selection_fig = px.bar(entry_selection, x="pace_band", y="share_pct", color="entry_type",
                      hover_data=["entries"], category_orders={"pace_band": ["Slower", "Middle", "Faster"]}, labels={"share_pct":"Entry selection (%)", "pace_band":"Preceding pace"},
                      title="5v5 entry choices at different approach paces")
selection_fig.update_layout(template="plotly_white",height=450)
selection_fig.show()


## 7.6. Review team context at 5v5

Retain counts and games when comparing teams at 5v5. These descriptive splits are not an adjusted causal analysis. Review actual sequences before interpreting a high-pace difference, especially if it reflects opponent strength, available support, or source-specific tagging.

Assess recorded-time sensitivity and inspect inferred sequence boundaries before making recommendations. A useful result identifies an entry method that remains productive at higher pace, or a failure pattern to review, rather than claiming every team should always play faster.


In [ ]:
team_entry_summary = analysis_entries.groupby(
    ["team_id", "team_name", "entry_type", "pace_tertile"], observed=True
).agg(entries=("event_id", "size"), games=("game_id", "nunique"), success_rate=("outcome_success", "mean")).reset_index()
display(team_entry_summary.round(3))

REVIEW_ENTRY_TYPE = "Dumped"
REVIEW_TERTILE = 3
review_entries = analysis_entries.loc[
    analysis_entries["entry_type"].eq(REVIEW_ENTRY_TYPE)
    & analysis_entries["pace_tertile"].eq(REVIEW_TERTILE)
]
display(review_entries[["game_id", "game_date", "team_name", "period", "clock_seconds",
    "event_id", "possession_id", "preceding_pace_ft_s", "preceding_movement_seconds",
    "situation", "outcome_success", "outcome_event_id", "recovered_by_team_id"]].head(20))


## 7.7. Compare average shot distance by entry pace

Compare average shot-attempt distance to the attacking net for slower, middle, and faster entries, separately for carried, played, and dumped entries. Retain all subsequent shot attempts within five seconds, before opponent control, a restart, another entry, a goal, or a manpower change. A goal counts as an attempt and ends the search.

Use entries with measurable preceding sequence pace and complete five-second follow-up. Count each shot once. Coordinates use the shooting team's attacking perspective, with the attacking net center at `(189, 42.5)` on the normalized 200-by-85-foot rink. Calculate Euclidean distance for each located attempt, then average those distances; do not measure distance from an average location.

Invalid or missing locations are excluded and reported. Groups with no located shots have no average, rather than a zero distance. The mean is shot-weighted and conditional on generating a shot; it does not measure overall entry success or expected goals. Display shot and entry counts alongside the mean. No uncertainty intervals are shown.


In [38]:
shot_rows = []
for entry in analysis_entries.itertuples(index=False):
    following = period_events[(entry.game_id, entry.period)]
    following = following.loc[
        following["event_id"].gt(entry.event_id)
        & following["modeled_clock_seconds"].le(entry.modeled_clock_seconds)
        & following["modeled_clock_seconds"].ge(entry.modeled_clock_seconds - OUTCOME_SECONDS)
    ]
    for event in following.itertuples(index=False):
        if not event.is_5v5 or event.event in {"Faceoff Win", "Penalty Shot Goal", "Zone Entry"}:
            break
        if event.event in control_events and event.team_id != entry.team_id:
            break
        if event.event in {"Shot", "Goal"} and event.team_id == entry.team_id:
            shot_rows.append({
                "entry_event_id": entry.event_id, "shot_event_id": event.event_id,
                "entry_type": entry.entry_type, "pace_band": entry.pace_band,
                "x": event.x, "y": event.y, "event": event.event,
            })
        if event.event == "Goal":
            break

entry_shots = pd.DataFrame(shot_rows, columns=[
    "entry_event_id", "shot_event_id", "entry_type", "pace_band", "x", "y", "event",
])
assert not entry_shots["shot_event_id"].duplicated().any()
valid_locations = (entry_shots["x"].between(0, 200) & entry_shots["y"].between(0, 85)).fillna(False)
located_shots = entry_shots.loc[valid_locations].copy()
print(f"Shot attempts: {len(entry_shots)} | located: {len(located_shots)} | excluded locations: {(~valid_locations).sum()}")
shot_distance_coverage = analysis_entries.groupby(["entry_type", "pace_band"]).size().rename("entries").to_frame()
shot_distance_coverage["shot_attempts"] = entry_shots.groupby(["entry_type", "pace_band"]).size()
shot_distance_coverage["located_attempts"] = located_shots.groupby(["entry_type", "pace_band"]).size()
shot_distance_coverage[["shot_attempts", "located_attempts"]] = shot_distance_coverage[["shot_attempts", "located_attempts"]].fillna(0).astype(int)
display(shot_distance_coverage)


Shot attempts: 903 | located: 903 | excluded locations: 0


entries  shot_attempts  located_attempts
entry_type pace_band                                          
Carried    Faster         706            355               355
           Middle         517            228               228
           Slower         425            202               202
Dumped     Faster         200              2                 2
           Middle         441              8                 8
           Slower         560             10                10
Played     Faster         132             58                58
           Middle          75             29                29
           Slower          48             11                11

In [39]:
NET_X_FT = 189
NET_Y_FT = 42.5
located_shots["distance_to_net_ft"] = np.hypot(
    located_shots["x"] - NET_X_FT, located_shots["y"] - NET_Y_FT
)
shot_distance_summary = shot_distance_coverage.join(
    located_shots.groupby(["entry_type", "pace_band"])["distance_to_net_ft"]
    .mean().rename("mean_distance_ft")
).reset_index()
display(shot_distance_summary.round(2))

shot_distance_fig = px.bar(
    shot_distance_summary, x="pace_band", y="mean_distance_ft", color="pace_band",
    facet_col="entry_type", text="mean_distance_ft",
    category_orders={"pace_band": ["Slower", "Middle", "Faster"],
                     "entry_type": ["Carried", "Played", "Dumped"]},
    color_discrete_map={"Slower": "#93c5fd", "Middle": "#3b82f6", "Faster": "#1d4ed8"},
    hover_data=["entries", "shot_attempts", "located_attempts"],
    labels={"pace_band": "Preceding sequence pace", "mean_distance_ft": "Average shot distance (ft)"},
    title=f"5v5 average shot distance | within {OUTCOME_SECONDS}s of entry",
)
shot_distance_fig.update_traces(texttemplate="%{y:.1f} ft", textposition="outside")
shot_distance_fig.update_yaxes(rangemode="tozero")
shot_distance_fig.update_layout(template="plotly_white", height=450, showlegend=False)
shot_distance_fig.show()


,entry_type,pace_band,entries,shot_attempts,located_attempts,mean_distance_ft
0,Carried,Faster,706,355,355,33.01
1,Carried,Middle,517,228,228,34.99
2,Carried,Slower,425,202,202,33.27
3,Dumped,Faster,200,2,2,29.63
4,Dumped,Middle,441,8,8,24.21
5,Dumped,Slower,560,10,10,27.08
6,Played,Faster,132,58,58,26.51
7,Played,Middle,75,29,29,33.88
8,Played,Slower,48,11,11,30.33
